In [15]:
import pandas as pd
import altair as alt


# Load csv
df_initial = pd.read_csv(
    "hospital-utilization-trends-by-health-category_2020-2024.csv",
    encoding="cp1252"
)

# Pre-process data
df = df_initial[
    df_initial["Facility Name"] == "Huntington Hospital"
].copy()

df["Date"] = pd.to_datetime(
    df["Date"],
    format="%y-%b"
)

df["Year"] = df["Date"].dt.year.astype(str)


# Create my selections:

# Year selector
year_select = alt.selection_point(
    fields=["Year"],
    empty="none"
)
# Department selector
setting_select = alt.selection_point(
    fields=["Setting"],
    empty="none"
)


# Chart 1: Year Selector 
years = (
    alt.Chart(df)
    .transform_aggregate(
        total_encounters="sum(Count)",
        groupby=["Year"]
    )
    .mark_bar()
    .encode(
        y=alt.Y(
            "Year:N",
            title="Year",
            sort="ascending"
        ),

        x=alt.X(
            "total_encounters:Q",
            title="Total Patient Encounters"
        ),

        color=alt.condition(
            year_select,
            alt.value("#4C78A8"),
            alt.value("lightgray")
        ),

        tooltip=[
            alt.Tooltip(
                "Year:N"
            ),
            alt.Tooltip(
                "total_encounters:Q",
                title="Encounters",
                format=","
            )
        ]
    )
    .properties(
        title="(Select Year Below)",
        width=800,
        height=150
    )
    .add_params(year_select)
)



# Chart 2: Department

settings = (
    alt.Chart(df)
    .transform_filter(year_select)
    .transform_aggregate(
        total_encounters="sum(Count)",
        groupby=["Setting"]
    )
    .mark_bar()
    .encode(
        y=alt.Y(
            "Setting:N",
            sort="-x",
            title="Hospital Department"
        ),

        x=alt.X(
            "total_encounters:Q",
            title="Total Patient Encounters"
        ),

        # color=alt.condition(
        #     setting_select,
        #     alt.value("#4C78A8"),
        #     alt.value("lightgray")
        # ),

        color=alt.Color(
            "Setting:N",
            scale=alt.Scale(
                domain=[
                    "Inpatient",
                    "Emergency Department",
                    "Ambulatory Surgery"
                ],
            range=[
                "#D08C60",
                "#A65A6A",
                "#6FA58A"
            ]
            ),
            legend=None
        ),        

        tooltip=[
            alt.Tooltip(
                "Setting:N",
                title="Department"
            ),
            alt.Tooltip(
                "total_encounters:Q",
                title="Encounters",
                format=","
            )
        ]
    )
    .properties(
        title="Encounters by Department",
        width=400,
        height=200
    )
    .add_params(setting_select)
)



# Chart 3: Disease Categories
categories = (
    alt.Chart(df)
    .transform_filter(year_select)
    .transform_filter(setting_select)
    .transform_aggregate(
        total_encounters="sum(Count)",
        groupby=["Category"]
    )
    .mark_bar()
    .encode(
        y=alt.Y(
            "Category:N",
            sort="-x",
            title="Disease Category"
        ),

        x=alt.X(
            "total_encounters:Q",
            title="Total Patient Encounters"
        ),

        tooltip=[
            alt.Tooltip(
                "Category:N",
                title="Disease Category"
            ),
            alt.Tooltip(
                "total_encounters:Q",
                title="Encounters",
                format=","
            )
        ]
    )
    .properties(
        title="Encounters by Disease Categories",
        width=400,
        height=450
    )
)


# Combine:
final_chart = (
    alt.vconcat(
        years,
        alt.hconcat(
            settings,
            categories
        ),
        spacing=100
    )
    .properties(
        title={
            "text": "Utilization Trends at Huntington Hospital (2020-2024)",
            "fontSize": 22,
            "anchor": "middle",
            "offset": 15            
        }
    )
)


final_chart

final_chart.save('index.html')

C:\Users\kcch2\AppData\Local\Temp\ipykernel_12680\1967956035.py:6: DtypeWarning: Columns (0: System) have mixed types. Specify dtype option on import or set low_memory=False.
  df_initial = pd.read_csv(
